# Bayesian Modeling

Purpose:
Estimate latent working-memory precision using hierarchical Bayesian models.

The primary model assumes:

error ~ VonMises(0, κ)

where precision κ changes with memory load:

log(κ) = participant_precision + load_effect × setsize

The analysis evaluates:
1. Whether memory load affects precision.
2. Whether participants differ in baseline precision.
3. Whether degradation is nonlinear across set sizes.

In [1]:
import sys
from pathlib import Path
import gc

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

import jax
import numpyro

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RANDOM_SEED = 1234

from src.models import (
    build_null_model,
    build_hierarchical_model,
    build_nonlinear_model,
)

from src.visualization import (
    save_figure,
    plot_prior_predictive,
    plot_trace,
    plot_posterior,
    plot_posterior_predictive,
    plot_setsize_posterior_predictive,
    plot_rank,
    plot_participant_ppc,
    plot_residuals,
    plot_participant_influence,
    plot_posterior_stability,
)

In [2]:
DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "modeling-data.csv")

TABLES_DIR = (PROJECT_ROOT / "results" / "tables")
FIGURES_DIR = (PROJECT_ROOT / "results" / "figures")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Create Modeling Dataset

RAW_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "vandenberg12-clean.csv")

MODEL_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "modeling-data.csv")

model_columns = [
    "id",
    "experiment",
    "trial",
    "setsize",
    "errorrad",
]

model_df = (
    pd.read_csv(RAW_DATA_PATH)
    [model_columns]
    .copy()
)

# Validate

print(model_df.head())
print(model_df.info())

# Check missing values

print(model_df.isna().sum())

# Save modeling dataset

model_df.to_csv(
    MODEL_DATA_PATH,
    index=False,
)

   id experiment  trial  setsize  errorrad
0   1       Exp1      1        3  0.069813
1   1       Exp1      2        4  0.069813
2   1       Exp1      3        2  0.314159
3   1       Exp1      4        1  0.139626
4   1       Exp1      5        3  0.139626
<class 'pandas.DataFrame'>
RangeIndex: 37824 entries, 0 to 37823
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          37824 non-null  int64  
 1   experiment  37824 non-null  str    
 2   trial       37824 non-null  int64  
 3   setsize     37824 non-null  int64  
 4   errorrad    37824 non-null  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 1.4 MB
None
id            0
experiment    0
trial         0
setsize       0
errorrad      0
dtype: int64


In [4]:
# Load Modeling Dataset

df = pd.read_csv(DATA_PATH)

print(df.head())

print(df.shape)

print(df.dtypes)

print(df.isna().sum())

   id experiment  trial  setsize  errorrad
0   1       Exp1      1        3  0.069813
1   1       Exp1      2        4  0.069813
2   1       Exp1      3        2  0.314159
3   1       Exp1      4        1  0.139626
4   1       Exp1      5        3  0.139626
(37824, 5)
id              int64
experiment        str
trial           int64
setsize         int64
errorrad      float64
dtype: object
id            0
experiment    0
trial         0
setsize       0
errorrad      0
dtype: int64


In [5]:
# Prepare Variables

participant_codes, participants = pd.factorize(df["id"])

df["participant_idx"] = (participant_codes)

n_participants = len(participants)

print(n_participants)

# Prepare arrays

error = df["errorrad"].values

setsize = df["setsize"].values

participant_idx = (df["participant_idx"].values)

13


In [6]:
# Prior Predictive Check

model = build_hierarchical_model(
    error=error,
    setsize=setsize,
    participant_idx=participant_idx,
    n_participants=n_participants,
)

# Sample priors

with model:
    prior_predictive = pm.sample_prior_predictive()

# Inspect prior predictive

fig = plot_prior_predictive(prior_predictive)

save_figure(
    fig,
    FIGURES_DIR,
    "prior-predictive-check",
)

plt.show()

Sampling: [alpha, alpha_participant, beta, error, sigma_alpha]


In [7]:
# Null Model

null_model = build_null_model(
    error=error,
    participant_idx=participant_idx,
    n_participants=n_participants,
)

# Sample

with null_model:
    idata_null = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=4,
        target_accept=0.95,
        init="adapt_diag",
        nuts_sampler="numpyro",
        idata_kwargs={"log_likelihood": True},
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
    )

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [8]:
# Diagnostics

null_summary = az.summary(idata_null)

print(null_summary)

# Save Table

null_summary.to_csv(
    TABLES_DIR
    / "null-model-summary.csv"
)

                        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
alpha                  0.695  0.094   0.530    0.875      0.001    0.002   
alpha_participant[0]   0.917  0.018   0.883    0.951      0.000    0.000   
alpha_participant[1]   0.610  0.019   0.574    0.646      0.000    0.000   
alpha_participant[2]   0.342  0.021   0.301    0.380      0.000    0.000   
alpha_participant[3]   0.665  0.018   0.629    0.698      0.000    0.000   
alpha_participant[4]   0.421  0.020   0.385    0.462      0.000    0.000   
alpha_participant[5]   1.023  0.019   0.987    1.056      0.000    0.000   
alpha_participant[6]   0.760  0.029   0.706    0.815      0.000    0.000   
alpha_participant[7]   0.378  0.033   0.317    0.438      0.000    0.001   
alpha_participant[8]   0.639  0.030   0.586    0.696      0.000    0.000   
alpha_participant[9]   0.207  0.036   0.144    0.281      0.000    0.001   
alpha_participant[10]  1.040  0.029   0.985    1.094      0.000    0.000   
alpha_partic

In [9]:
# Primary Hierarchical Model

hierarchical_model = build_hierarchical_model(
    error=error,
    setsize=setsize,
    participant_idx=participant_idx,
    n_participants=n_participants,
)

In [10]:
# Sampling

with hierarchical_model:
    idata_hierarchical = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=4,
        target_accept=0.95,
        init="adapt_diag",
        nuts_sampler="numpyro",
        idata_kwargs={"log_likelihood": True},
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
    )

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [11]:
# Diagnostics

hierarchical_summary = az.summary(idata_hierarchical)

print(hierarchical_summary)

print(hierarchical_summary["r_hat"])

# Save Table

hierarchical_summary.to_csv(
    TABLES_DIR
    / "hierarchical-model-summary.csv"
)

                        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
alpha                  0.827  0.103   0.619    1.017      0.002    0.002   
beta                  -0.336  0.004  -0.343   -0.329      0.000    0.000   
alpha_participant[0]   1.088  0.021   1.047    1.125      0.000    0.000   
alpha_participant[1]   0.758  0.021   0.720    0.800      0.000    0.000   
alpha_participant[2]   0.382  0.022   0.342    0.425      0.000    0.000   
alpha_participant[3]   0.879  0.021   0.840    0.920      0.000    0.000   
alpha_participant[4]   0.556  0.022   0.519    0.600      0.000    0.000   
alpha_participant[5]   1.287  0.020   1.250    1.326      0.000    0.000   
alpha_participant[6]   0.839  0.033   0.779    0.901      0.000    0.001   
alpha_participant[7]   0.487  0.035   0.425    0.558      0.000    0.001   
alpha_participant[8]   0.785  0.034   0.722    0.849      0.000    0.001   
alpha_participant[9]   0.300  0.036   0.230    0.366      0.000    0.001   
alpha_partic

In [12]:
# Trace Plots

fig = plot_trace(
    idata_hierarchical,
    var_names=["alpha", "beta", "sigma_alpha"],
)

save_figure(
    fig,
    FIGURES_DIR,
    "hierarchical-traceplots",
)

plt.show()

In [13]:
# Posterior Analysis

load_effect_summary = az.summary(
    idata_hierarchical,
    var_names=["beta"],
    stat_focus="median",
)

# Save Table

load_effect_summary.to_csv(
    TABLES_DIR
    / "load-effect-summary.csv"
)

In [14]:
# Posterior Probability of a Negative Load Effect

beta_samples = (
    idata_hierarchical.posterior["beta"]
    .values
    .ravel()
)

prob_negative = np.mean(beta_samples < 0)

print(f"P(beta < 0) = {prob_negative:.4f}")

P(beta < 0) = 1.0000


In [15]:
# Posterior Distributions

fig = plot_posterior(
    idata_hierarchical,
    var_names=["alpha", "beta", "sigma_alpha"],
)

save_figure(
    fig,
    FIGURES_DIR,
    "posterior-distributions",
)

plt.show()

In [16]:
# Participant Variability

participant_variability_summary = az.summary(idata_hierarchical, var_names=["sigma_alpha"])

# Save Table

participant_variability_summary.to_csv(
    TABLES_DIR
    / "participant-variability-summary.csv"
)

In [17]:
# Nonlinear Model

nonlinear_model = build_nonlinear_model(
    error=error,
    setsize=setsize,
    participant_idx=participant_idx,
    n_participants=n_participants,
)

In [18]:
# Sampling

with nonlinear_model:
    idata_nonlinear = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        cores=4,
        target_accept=0.95,
        init="adapt_diag",
        nuts_sampler="numpyro",
        idata_kwargs={"log_likelihood": True},
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
    )

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

In [19]:
# Diagnostics

nonlinear_summary = az.summary(idata_nonlinear)

print(nonlinear_summary)

# Check worst convergence diagnostics

print(
    nonlinear_summary[["r_hat", "ess_bulk", "ess_tail"]]
        .sort_values("r_hat", ascending=False,)
        .head(10)
)

# Save Table

nonlinear_summary.to_csv(
    TABLES_DIR
    / "nonlinear-model-summary.csv"
)

                        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
alpha                  0.864  0.102   0.684    1.067      0.003    0.003   
alpha_offset[0]        0.719  0.325   0.092    1.298      0.010    0.008   
alpha_offset[1]       -0.200  0.291  -0.800    0.302      0.009    0.007   
alpha_offset[2]       -1.298  0.397  -2.048   -0.564      0.013    0.008   
alpha_offset[3]        0.162  0.292  -0.385    0.698      0.009    0.007   
alpha_offset[4]       -0.745  0.330  -1.389   -0.157      0.011    0.007   
alpha_offset[5]        1.293  0.397   0.575    2.051      0.013    0.009   
alpha_offset[6]        0.004  0.299  -0.551    0.556      0.009    0.007   
alpha_offset[7]       -0.981  0.361  -1.659   -0.312      0.011    0.007   
alpha_offset[8]       -0.133  0.299  -0.713    0.406      0.009    0.007   
alpha_offset[9]       -1.539  0.437  -2.359   -0.716      0.014    0.008   
alpha_offset[10]       0.988  0.361   0.336    1.675      0.011    0.008   
alpha_offset

In [20]:
# Compute PSIS-LOO for Null Model

loo_null = az.loo(idata_null)

In [21]:
# Compute PSIS-LOO for Hierarchical Model

loo_hierarchical = az.loo(idata_hierarchical)

In [22]:
# Compute PSIS-LOO for Nonlinear Model

loo_nonlinear = az.loo(idata_nonlinear)

In [26]:
# Pareto-k summary

for name, loo in [
    ("Null model", loo_null),
    ("Hierarchical model", loo_hierarchical),
    ("Nonlinear model", loo_nonlinear),
]:
    k = loo.pareto_k

    print(f"\n{name}")
    print(f"Maximum Pareto-k: {k.max().item():.3f}")
    print(f"Pareto-k > 0.7: {(k > 0.7).sum().item()}")
    print(f"Pareto-k > 1.0: {(k > 1.0).sum().item()}")


Null model
Maximum Pareto-k: 0.199
Pareto-k > 0.7: 0
Pareto-k > 1.0: 0

Hierarchical model
Maximum Pareto-k: 0.185
Pareto-k > 0.7: 0
Pareto-k > 1.0: 0

Nonlinear model
Maximum Pareto-k: 0.264
Pareto-k > 0.7: 0
Pareto-k > 1.0: 0


In [27]:
# Model Comparison

comparison = az.compare(
    {
        "Null": loo_null,
        "Hierarchical": loo_hierarchical,
        "Nonlinear": loo_nonlinear,
    }
)

print(comparison)

# Save Table

comparison.to_csv(
    TABLES_DIR / "model-comparison.csv"
)

              rank      elpd_loo      p_loo    elpd_diff    weight  \
Nonlinear        0 -42771.486772  34.257227     0.000000  0.871040   
Hierarchical     1 -42938.841220  25.295781   167.354447  0.069853   
Null             2 -47830.952559  22.030372  5059.465786  0.059108   

                      se         dse  warning scale  
Nonlinear     207.461238    0.000000    False   log  
Hierarchical  209.058700   24.306341    False   log  
Null          204.048575  111.383093    False   log  


In [28]:
# Posterior Predictive Checks

with hierarchical_model:
    posterior_predictive = (pm.sample_posterior_predictive(idata_hierarchical))

# Overall Error Distribution:

fig = plot_posterior_predictive(posterior_predictive)

save_figure(
    fig,
    FIGURES_DIR,
    "posterior-predictive-error",
)

plt.show()

Sampling: [error]


Output()

F:\doc-hub\education\psychology\projects\research\working-memory-bayesian-model\src\visualization.py:483: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.tight_layout()


In [29]:
# Set Size Posterior Predictive Check

# Observed mean error by set size

observed_means = (
    df.groupby("setsize")["errorrad"]
    .mean()
    .sort_index()
)

# Posterior predictive samples

posterior_errors = (
    posterior_predictive
    .posterior_predictive["error"]
    .stack(sample=("chain", "draw"))
    .transpose("sample", "error_dim_0")
    .values
)

set_sizes = np.sort(df["setsize"].unique())

predicted_means = []

# Mean predicted error for every posterior draw

for s in set_sizes:

    mask = df["setsize"].values == s

    predicted_means.append(
        posterior_errors[:, mask].mean(axis=1)
    )

predicted_means = np.asarray(predicted_means)

# Posterior summaries

predicted_mean = predicted_means.mean(axis=1)

predicted_hdi = np.percentile(predicted_means, [2.5, 97.5], axis=1,).T

# Figure

fig = plot_setsize_posterior_predictive(
    set_sizes=set_sizes,
    observed_means=observed_means.values,
    predicted_mean=predicted_mean,
    predicted_hdi=predicted_hdi,
)

save_figure(
    fig,
    FIGURES_DIR,
    "posterior-predictive-setsize",
)

plt.show()

In [30]:
# Rank Plots

fig = plot_rank(
    idata_nonlinear,
    var_names=["alpha", "sigma_alpha", "beta_setsize"],
)

save_figure(
    fig,
    FIGURES_DIR,
    "nonlinear-rank-plots",
)

plt.show()

In [31]:
# Participant-Level Posterior Predictive Check

posterior_errors = (
    posterior_predictive
    .posterior_predictive["error"]
    .stack(sample=("chain", "draw"))
    .transpose("sample", "error_dim_0")
    .values
)

observed = []
predicted = []

participant_ids = np.sort(df["id"].unique())

for participant in participant_ids:

    mask = df["id"].values == participant

    observed.append(
        error[mask].mean()
    )

    predicted.append(
        posterior_errors[:, mask].mean(axis=1).mean()
    )

fig = plot_participant_ppc(observed, predicted)

save_figure(
    fig,
    FIGURES_DIR,
    "participant-level-ppc",
)

plt.show()

In [32]:
# Residual Diagnostics

predicted_error = posterior_errors.mean(axis=0)

residuals = error - predicted_error

fig = plot_residuals(setsize, residuals)

save_figure(
    fig,
    FIGURES_DIR,
    "residuals-by-setsize",
)

plt.show()

Robustness / Sensitivity Analyses

In [33]:
# Prior Sensitivity

prior_settings = {
    "Primary": {
        "alpha_sd": 2.0,
        "beta_sd": 1.0,
        "sigma_alpha_sd": 1.0,
    },
    "Weak": {
        "alpha_sd": 4.0,
        "beta_sd": 2.0,
        "sigma_alpha_sd": 2.0,
    },
    "Strong": {
        "alpha_sd": 1.0,
        "beta_sd": 0.5,
        "sigma_alpha_sd": 0.5,
    },
}

prior_results = []

for name, priors in prior_settings.items():

    model = build_hierarchical_model(
        error=error,
        setsize=setsize,
        participant_idx=participant_idx,
        n_participants=n_participants,
        **priors,
    )

    with model:

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            nuts_sampler="numpyro",
            random_seed=RANDOM_SEED,
            progressbar=False,
        )

    summary = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta",
            "sigma_alpha",
        ],
    )

    summary["prior"] = name

    prior_results.append(summary)

prior_results = (
    pd.concat(prior_results)
    .reset_index(names="parameter")
)

print(prior_results)

prior_results.to_csv(
    TABLES_DIR / "prior-sensitivity.csv",
    index=False,
)

     parameter   mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
0        alpha  0.827  0.103   0.619    1.017      0.002    0.002    3903.0   
1         beta -0.336  0.004  -0.343   -0.329      0.000    0.000    6604.0   
2  sigma_alpha  0.368  0.086   0.236    0.529      0.002    0.002    3823.0   
3        alpha  0.828  0.104   0.642    1.031      0.002    0.002    4107.0   
4         beta -0.336  0.004  -0.343   -0.329      0.000    0.000    5814.0   
5  sigma_alpha  0.368  0.087   0.225    0.521      0.002    0.002    3372.0   
6        alpha  0.819  0.100   0.616    0.998      0.002    0.002    3969.0   
7         beta -0.336  0.004  -0.343   -0.329      0.000    0.000    6232.0   
8  sigma_alpha  0.357  0.077   0.230    0.499      0.001    0.002    3998.0   

   ess_tail  r_hat    prior  
0    2691.0    1.0  Primary  
1    3217.0    1.0  Primary  
2    2676.0    1.0  Primary  
3    2645.0    1.0     Weak  
4    2589.0    1.0     Weak  
5    2624.0    1.0     Weak  

In [34]:
# Sampling Robustness

sampling_settings = {
    "Primary": dict(draws=2000, tune=1000, target_accept=0.95),
    "LongTune": dict(draws=2000, tune=2000, target_accept=0.95),
    "HighAccept": dict(draws=2000, tune=1000, target_accept=0.99),
}

sampling_results = []

for name, settings in sampling_settings.items():

    model = build_hierarchical_model(
        error=error,
        setsize=setsize,
        participant_idx=participant_idx,
        n_participants=n_participants,
    )

    with model:

        idata = pm.sample(
            chains=4,
            cores=4,
            nuts_sampler="numpyro",
            random_seed=RANDOM_SEED,
            progressbar=False,
            **settings,
        )

    summary = az.summary(
        idata,
        var_names=["alpha", "beta", "sigma_alpha"],
    )

    summary["configuration"] = name

    sampling_results.append(summary)

sampling_results = pd.concat(sampling_results)

sampling_results.to_csv(
    TABLES_DIR / "sampling-sensitivity.csv"
)

In [35]:
# Participant Influence

participant_summary = (
    az.summary(
        idata_hierarchical,
        var_names=["alpha_participant"],
    )
    .reset_index()
)

participant_summary.to_csv(
    TABLES_DIR / "participant-influence.csv",
    index=False,
)

participant_influence = pd.DataFrame(
    {
        "participant": participant_ids,
        "mean_error": observed,
        "predicted_error": predicted,
    }
)


participant_influence.to_csv(
    TABLES_DIR / "participant-influence.csv",
    index=False,
)


fig = plot_participant_influence(
    participant_influence
)


save_figure(
    fig,
    FIGURES_DIR,
    "participant-influence",
)

In [36]:
# Experiment Robustness

experiment_results = []

for experiment in sorted(df["experiment"].unique()):

    subset = df[df["experiment"] == experiment]

    participant_codes, _ = pd.factorize(subset["id"])

    model = build_hierarchical_model(
        error=subset["errorrad"].values,
        setsize=subset["setsize"].values,
        participant_idx=participant_codes,
        n_participants=len(np.unique(participant_codes)),
    )

    with model:

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            nuts_sampler="numpyro",
            random_seed=RANDOM_SEED,
            progressbar=False,
        )

    summary = az.summary(
        idata,
        var_names=["beta"],
    )

    summary["experiment"] = experiment

    experiment_results.append(summary)

experiment_results = pd.concat(experiment_results)

experiment_results.to_csv(
    TABLES_DIR / "experiment-robustness.csv"
)

In [37]:
# Posterior Stability

seeds = [42, 123, 456]

stability_results = []

for seed in seeds:

    model = build_hierarchical_model(
        error=error,
        setsize=setsize,
        participant_idx=participant_idx,
        n_participants=n_participants,
    )

    with model:

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            nuts_sampler="numpyro",
            random_seed=seed,
            progressbar=False,
        )

    summary = az.summary(
        idata,
        var_names=["alpha", "beta", "sigma_alpha"],
    )

    summary["seed"] = seed

    stability_results.append(summary)


stability_results = pd.concat(stability_results)

In [38]:
# Convert index to column

stability_results = (
    stability_results
    .reset_index()
    .rename(columns={"index": "parameter"})
)

# Save table

stability_results.to_csv(
    TABLES_DIR / "posterior-stability.csv",
    index=False,
)

In [39]:
# Posterior Stability Figure

stability_results.to_csv(
    TABLES_DIR / "posterior-stability.csv",
    index=False,
)


fig = plot_posterior_stability(
    stability_results
)


save_figure(
    fig,
    FIGURES_DIR,
    "posterior-stability",
)

plt.show()